# VCF analysis

## Summary and objectives

This analysis will describe the VCF file originating from the exome-capture sequencing of the PeaMUST _Vicia faba_ collection. 

Statistics will be produced from the **raw VCF file** and from the **filtered VCF**. Filtering is based on: 
- MAF (Minor Allele Frequency) rate, 
- Mean read depth, 
- SNP density along chromosomes


**Metrics** to be generated on the raw VCF file and on filtered VCF files:    
- Number of variants per chromosomes,
- Mean depth per variant, 
- Heterozygosis rate,
- etc.



Environment used to analyse the VCF file is exported: 

In [11]:
import subprocess as sp
shell_cmd = "mamba env export -n vcf > ./vcf_analysis_environment.yaml"
sp.run(shell_cmd, shell=True, check=True)

CompletedProcess(args='mamba env export -n vcf > ./vcf_analysis_environment.yaml', returncode=0)

# 1. Analysis on the raw VCF file

## 1.1 Using bcftools 

A nice cheat sheet around `bcftools` is available [here](https://gist.github.com/elowy01/93922762e131d7abd3c7e8e166a74a0b)

In [12]:
# Test and real VCF files
#snp_file = "./test100kSNPs.vcf"
snp_file = "../../../../00.info/03.SNP_data_exome_capture/vfaba_hedin_peamust_SNP_GATK.vcf"
stat_file = "./vcf_stats.txt"

shell_cmd = "bcftools stats " + snp_file + " > " + stat_file

sp.run(shell_cmd, shell=True, check=True)

CompletedProcess(args='bcftools stats ../../../../00.info/03.SNP_data_exome_capture/vfaba_hedin_peamust_SNP_GATK.vcf > ./vcf_stats.txt', returncode=0)

### Number of total variants overall and per chromosomes

In [13]:
! head -n 50 vcf_stats.txt | grep "^SN"

SN	0	number of samples:	248
SN	0	number of records:	6425454
SN	0	number of no-ALTs:	0
SN	0	number of SNPs:	6425454
SN	0	number of MNPs:	0
SN	0	number of indels:	0
SN	0	number of others:	0
SN	0	number of multiallelic sites:	606446
SN	0	number of multiallelic SNP sites:	284919


There are **248 samples** and **6 425 454 SNPs** in total in the unfiltered VCF file.  
There are no InDels as these are filtered out most likely. 

**606 446** sites have multiple alternate alleles (around 10%). 

## 1.2. Analysis using VCFtools

In [14]:
import subprocess
outdir_vcftools_data = "vcftools_data/"
shell_cmd = "mkdir -p " + outdir_vcftools_data
subprocess.call(shell_cmd, shell=True)

0

### 1.2.1 Allele frequency

shell_cmd = "vcftools --freq --vcf " + outdir_vcftools_data + snp_file + " --out " + "./allele_frequency.freq"
sp.run(shell_cmd, shell=True, check=True)

In [15]:
### 1.2.2 

# 2. Filtering the raw VCF file

## 2.1 Keep only biallelic SNPs

A biallelic site is a locus where you have only two alleles possible, the REF allele (same as genome assembly) and the Alt allele. 
A explanation is available here: https://gatk.broadinstitute.org/hc/en-us/articles/360035890771-Biallelic-vs-Multiallelic-sites 

In [16]:
shell_cmd = "bcftools view  --max-alleles 2 --with-header -Ou --output ./biallelic_SNPs.vcf " + snp_file 
sp.run(shell_cmd, shell=True, check=True)

CompletedProcess(args='bcftools view  --max-alleles 2 --with-header -Ou --output ./biallelic_SNPs.vcf ../../../../00.info/03.SNP_data_exome_capture/vfaba_hedin_peamust_SNP_GATK.vcf', returncode=0)

How many SNPs remaining? 

In [17]:
shell_cmd = "bcftools stats ./biallelic_SNPs.vcf | grep ^SN " 
sp.run(shell_cmd, shell=True, check=True)

SN	0	number of samples:	248
SN	0	number of records:	5819008
SN	0	number of no-ALTs:	0
SN	0	number of SNPs:	5819008
SN	0	number of MNPs:	0
SN	0	number of indels:	0
SN	0	number of others:	0
SN	0	number of multiallelic sites:	0
SN	0	number of multiallelic SNP sites:	0


CompletedProcess(args='bcftools stats ./biallelic_SNPs.vcf | grep ^SN ', returncode=0)

Keeping only biallelic SNPs reduces the number of SNPs from 6,425,454 to 5,819,008 SNPs (same results as Jonathan, see below)  

<img src="snp_different_filters.png" width=50%>